<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L1 · Sizing
Tamaño = riesgo_usd / |entrada-stop| con 50 trades de 10.000 al 1-2%. Replicas la columna tamano.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/riesgo/data/c4_l1.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c4_l1.csv'), Path('data/c4_l1.csv'), Path('c4_l1.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)

In [ ]:
df['riesgo_calc'] = df['capital'] * df['riesgo_pct'] / 100
df['tamano_calc'] = df['riesgo_calc'] / (df['entrada'] - df['stop']).abs()
print(df[['trade','riesgo_usd','riesgo_calc','tamano','tamano_calc']].head(8).to_string(index=False))
print(f"dif max tamano={ (df['tamano']-df['tamano_calc']).abs().max():.2e}")

In [ ]:
racha1 = 1 - (1-0.01)**10
racha2 = 1 - (1-0.02)**10
print(f'10 stops seguidos al 1%: -{racha1:.2%}   al 2%: -{racha2:.2%}')
print(df.groupby('riesgo_pct')['tamano'].mean().round(4).to_string())

In [ ]:
# Chequeo automático
assert ((df['riesgo_calc'] - df['riesgo_usd']).abs().max() < 1e-6), 'riesgo_usd no replica'
assert ((df['tamano'] - df['tamano_calc']).abs().max() < 1e-6), 'tamano no replica la fórmula'
assert abs((1 - (1-0.01)**10) - 0.0956) < 1e-3
print('OK: sizing verificado en 50 trades')